# CENG 463 – HW3: CIFAR-10 Image Clustering
**Due Date:** 05.01.2026

As an **image clustering task**, you will work with the CIFAR-10 dataset, which contains images from 10 classes, including airplanes, cars, cats, and dogs. The goal is to use **unsupervised learning** techniques to group similar images automatically and visualize the results.

## 1. Dataset Overview (10 pts)
- Load the dataset using Keras (More information: [Keras CIFAR-10.](https://keras.io/api/datasets/cifar10/)):

    from keras.datasets import cifar10

   (x_train, y_train), (x_test, y_test) = cifar10.load_data()

- Show the dataset size and class names in a table.
- Display sample images from each class.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CIFAR-10 dataset (compatible with both old and new Keras/TensorFlow versions)
try:
    from keras.datasets import cifar10
except ImportError:
    from tensorflow.keras.datasets import cifar10

# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Dataset size information
dataset_info = {
    'Dataset': ['Training Set', 'Test Set', 'Total', 'Image Shape', 'Number of Classes'],
    'Size/Value': [
        f'{x_train.shape[0]} images',
        f'{x_test.shape[0]} images',
        f'{x_train.shape[0] + x_test.shape[0]} images',
        f'{x_train.shape[1]}x{x_train.shape[2]}x{x_train.shape[3]}',
        len(class_names)
    ]
}

df_info = pd.DataFrame(dataset_info)
print("\n=== CIFAR-10 Dataset Information ===")
print(df_info.to_string(index=False))

# Class names table
class_table = pd.DataFrame({
    'Class ID': range(10),
    'Class Name': class_names
})
print("\n=== Class Names ===")
print(class_table.to_string(index=False))

# Display sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold')

for i, class_name in enumerate(class_names):
    # Find first image of this class
    idx = np.where(y_train == i)[0][0]
    ax = axes[i // 5, i % 5]
    ax.imshow(x_train[idx])
    ax.set_title(f'{i}: {class_name}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 1. Dataset Overview (10 pts)
- Load the dataset using Keras (More information: [Keras CIFAR-10.](https://keras.io/api/datasets/cifar10/)):

    from keras.datasets import cifar10

   (x_train, y_train), (x_test, y_test) = cifar10.load_data()

- Show the dataset size and class names in a table.
- Display sample images from each class.


## 2. Feature Extraction and PCA (30 pts)
- Convert each image from 32×32×3 into a 3072-dimensional vector.
- Scale the vectors using StandardScaler.
- Apply PCA to reduce the dimensionality from 3072 → 50.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Use training data for clustering
X = x_train

# Step 1: Convert images from 32x32x3 to 3072-dimensional vectors
print("Original shape:", X.shape)
X_flattened = X.reshape(X.shape[0], -1)
print(f"Flattened shape: {X_flattened.shape}")
print(f"Each image is now a {X_flattened.shape[1]}-dimensional vector")

# Step 2: Scale the vectors using StandardScaler
print("\nScaling features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flattened)
print(f"Scaled data shape: {X_scaled.shape}")
print(f"Mean: {X_scaled.mean():.6f}, Std: {X_scaled.std():.6f}")

# Step 3: Apply PCA to reduce dimensionality from 3072 → 50
print("\nApplying PCA to reduce dimensions from 3072 → 50...")
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA-transformed shape: {X_pca.shape}")
print(f"Explained variance ratio (first 10 components): {pca.explained_variance_ratio_[:10]}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.4f}")

# Visualize explained variance
plt.figure(figsize=(12, 5))

# Plot 1: Explained variance by component
plt.subplot(1, 2, 1)
plt.bar(range(1, 51), pca.explained_variance_ratio_, alpha=0.7, color='steelblue')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance by Principal Component')
plt.grid(True, alpha=0.3)

# Plot 2: Cumulative explained variance
plt.subplot(1, 2, 2)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, 51), cumulative_variance, marker='o', linestyle='-', color='darkgreen')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance')
plt.grid(True, alpha=0.3)
plt.axhline(y=cumulative_variance[-1], color='r', linestyle='--', 
            label=f'50 components: {cumulative_variance[-1]:.4f}')
plt.legend()

plt.tight_layout()
plt.show()

print(f"\n✓ Feature extraction completed successfully!")
print(f"✓ Data reduced from {X_flattened.shape[1]} to {X_pca.shape[1]} dimensions")

**Note:** To avoid long computation times and crashes, we use **stratified sampling** to work with a smaller representative subset of the data while maintaining class proportions.


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# IMPORTANT: Use stratified sampling to reduce computation time
# This maintains class proportions while working with a smaller dataset
n_samples = 3000  # Optimal: 2500-3000 (fast) or 5000 (slower but more data)

print("="*70)
print("STRATIFIED SAMPLING FOR FASTER COMPUTATION")
print("="*70)
print(f"Original dataset size: {X_pca.shape[0]} samples")
print(f"Sampling {n_samples} samples (maintaining class proportions)...")

sss = StratifiedShuffleSplit(n_splits=1, train_size=n_samples, random_state=42)

for train_idx, _ in sss.split(X_pca, y_train):
    X_pca_sample = X_pca[train_idx]
    X_sample = X[train_idx]
    y_sample = y_train[train_idx]

print(f"✓ Sampled dataset size: {X_pca_sample.shape[0]} samples")
print(f"  This is {(n_samples/X_pca.shape[0])*100:.1f}% of the original data")
print(f"  Computation will be ~{(X_pca.shape[0]/n_samples):.0f}x faster!")

# Verify class distribution is maintained
print("\nClass distribution in sampled data:")
for i in range(10):
    count = np.sum(y_sample == i)
    print(f"  Class {i} ({class_names[i]:>10}): {count:>4} samples ({count/n_samples*100:.1f}%)")

print("="*70)


## 3. Clustering Methods (30 pts)
- Perform clustering on the PCA-transformed features using three Sklearn algorithms: K-Means, Agglomerative Clustering, DBSCAN.
- Visualize example images for each cluster.
- Provide short comments on the clusters.

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
import warnings
warnings.filterwarnings('ignore')

# Function to display sample images from clusters
def display_cluster_samples(X_original, labels, algorithm_name, n_clusters=10, samples_per_cluster=5):
    """Display sample images from each cluster"""
    fig, axes = plt.subplots(n_clusters, samples_per_cluster, figsize=(15, 3*n_clusters))
    fig.suptitle(f'{algorithm_name} - Sample Images from Each Cluster', fontsize=16, fontweight='bold')
    
    unique_labels = np.unique(labels)
    for i, cluster_id in enumerate(unique_labels):
        if i >= n_clusters:
            break
        # Get indices of images in this cluster
        cluster_indices = np.where(labels == cluster_id)[0]
        
        # Sample random images from this cluster
        sample_indices = np.random.choice(cluster_indices, 
                                         min(samples_per_cluster, len(cluster_indices)), 
                                         replace=False)
        
        for j, idx in enumerate(sample_indices):
            if n_clusters == 1:
                ax = axes[j]
            else:
                ax = axes[i, j]
            ax.imshow(X_original[idx])
            ax.axis('off')
            if j == 0:
                ax.set_ylabel(f'Cluster {cluster_id}\n({len(cluster_indices)} images)', 
                            fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

print("="*70)
print("CLUSTERING ANALYSIS")
print("="*70)

# 1. K-MEANS CLUSTERING
print("\n1. K-MEANS CLUSTERING")
print("-" * 70)
n_clusters = 10  # Same as number of classes

print(f"Performing K-Means clustering with k={n_clusters}...")
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10, max_iter=300, algorithm='elkan')
kmeans_labels = kmeans.fit_predict(X_pca_sample)

print(f"✓ K-Means completed!")
print(f"  Number of clusters: {n_clusters}")
print(f"  Inertia (within-cluster sum of squares): {kmeans.inertia_:.2f}")

# Display cluster distribution
unique, counts = np.unique(kmeans_labels, return_counts=True)
print(f"\n  Cluster distribution:")
for cluster_id, count in zip(unique, counts):
    print(f"    Cluster {cluster_id}: {count} images ({count/len(kmeans_labels)*100:.1f}%)")

# Display sample images
display_cluster_samples(X_sample, kmeans_labels, "K-Means")

print("\n  Comments on K-Means:")
print("  • K-Means creates relatively balanced clusters")
print("  • Each cluster contains a mix of different object types")
print("  • Works well for spherical clusters but may struggle with complex shapes")
print("  • Fast and scalable for large datasets")

# 2. AGGLOMERATIVE CLUSTERING
print("\n2. AGGLOMERATIVE CLUSTERING")
print("-" * 70)

print(f"Performing Agglomerative Clustering with {n_clusters} clusters...")
agg_clustering = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
agg_labels = agg_clustering.fit_predict(X_pca_sample)

print(f"✓ Agglomerative Clustering completed!")
print(f"  Number of clusters: {n_clusters}")

# Display cluster distribution
unique, counts = np.unique(agg_labels, return_counts=True)
print(f"\n  Cluster distribution:")
for cluster_id, count in zip(unique, counts):
    print(f"    Cluster {cluster_id}: {count} images ({count/len(agg_labels)*100:.1f}%)")

# Display sample images
display_cluster_samples(X_sample, agg_labels, "Agglomerative Clustering")

print("\n  Comments on Agglomerative Clustering:")
print("  • Creates hierarchical structure of clusters")
print("  • Can capture more complex cluster shapes than K-Means")
print("  • Cluster sizes may be more varied")
print("  • Good for discovering natural groupings in the data")

# 3. DBSCAN
print("\n3. DBSCAN (Density-Based Spatial Clustering)")
print("-" * 70)

# DBSCAN parameters - tuned for this dataset size
eps = 5.5
min_samples = 15

print(f"Performing DBSCAN with eps={eps}, min_samples={min_samples}...")
dbscan = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1, algorithm='ball_tree')
dbscan_labels = dbscan.fit_predict(X_pca_sample)

# Count clusters (excluding noise points labeled as -1)
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f"✓ DBSCAN completed!")
print(f"  Number of clusters found: {n_clusters_dbscan}")
print(f"  Number of noise points: {n_noise} ({n_noise/len(dbscan_labels)*100:.1f}%)")

# Display cluster distribution
unique, counts = np.unique(dbscan_labels, return_counts=True)
print(f"\n  Cluster distribution:")
for cluster_id, count in zip(unique, counts):
    if cluster_id == -1:
        print(f"    Noise: {count} images ({count/len(dbscan_labels)*100:.1f}%)")
    else:
        print(f"    Cluster {cluster_id}: {count} images ({count/len(dbscan_labels)*100:.1f}%)")

# Display sample images (limit to first 10 clusters)
n_display_clusters = min(10, n_clusters_dbscan)
if n_clusters_dbscan > 0:
    display_cluster_samples(X_sample, dbscan_labels, "DBSCAN", n_clusters=n_display_clusters)
else:
    print("  Note: No clusters found, only noise points")

print("\n  Comments on DBSCAN:")
print("  • Identifies dense regions as clusters")
print("  • Can find arbitrarily shaped clusters")
print("  • Automatically determines number of clusters")
print("  • Labels outliers as noise points")
print("  • Sensitive to epsilon and min_samples parameters")
print("  • May find fewer clusters than expected if parameters are not well-tuned")

print("\n" + "="*70)
print("CLUSTERING SUMMARY")
print("="*70)
print(f"K-Means:        {n_clusters} clusters")
print(f"Agglomerative:  {n_clusters} clusters")
print(f"DBSCAN:         {n_clusters_dbscan} clusters + {n_noise} noise points")
print("="*70)

## 4. Visualization and Analysis (30 pts)
- Use t-SNE to reduce PCA features to 2D for visualization.
- Create colored scatter plots for each clustering algorithm.
- Observe which classes are better grouped by each algorithm.

In [ ]:
from sklearn.manifold import TSNE

print("="*70)
print("t-SNE VISUALIZATION AND ANALYSIS")
print("="*70)

# Apply t-SNE to reduce PCA features to 2D
print("\nApplying t-SNE to reduce PCA features to 2D...")
print("Using optimized Barnes-Hut algorithm for faster computation...")

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=500, 
            learning_rate=200, init='random', verbose=1)
X_tsne = tsne.fit_transform(X_pca_sample)

print(f"✓ t-SNE completed!")
print(f"  Reduced from {X_pca_sample.shape[1]}D to {X_tsne.shape[1]}D")
print(f"  Final shape: {X_tsne.shape}")

# Create colored scatter plots for each clustering algorithm
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('t-SNE Visualization of Clustering Results', fontsize=18, fontweight='bold')

# Define color maps
colors_true = plt.cm.tab10(np.linspace(0, 1, 10))
colors_kmeans = plt.cm.tab10(np.linspace(0, 1, 10))
colors_agg = plt.cm.tab10(np.linspace(0, 1, 10))

# 1. True Labels (Ground Truth)
ax1 = axes[0, 0]
for i in range(10):
    mask = (y_sample.flatten() == i)
    ax1.scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
               c=[colors_true[i]], label=class_names[i], 
               alpha=0.6, s=10, edgecolors='none')
ax1.set_title('Ground Truth Labels', fontsize=14, fontweight='bold')
ax1.set_xlabel('t-SNE Component 1')
ax1.set_ylabel('t-SNE Component 2')
ax1.legend(loc='best', markerscale=2, fontsize=8, ncol=2)
ax1.grid(True, alpha=0.3)

# 2. K-Means Clustering
ax2 = axes[0, 1]
for i in range(10):
    mask = (kmeans_labels == i)
    ax2.scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
               c=[colors_kmeans[i]], label=f'Cluster {i}', 
               alpha=0.6, s=10, edgecolors='none')
ax2.set_title('K-Means Clustering', fontsize=14, fontweight='bold')
ax2.set_xlabel('t-SNE Component 1')
ax2.set_ylabel('t-SNE Component 2')
ax2.legend(loc='best', markerscale=2, fontsize=8, ncol=2)
ax2.grid(True, alpha=0.3)

# 3. Agglomerative Clustering
ax3 = axes[1, 0]
for i in range(10):
    mask = (agg_labels == i)
    ax3.scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
               c=[colors_agg[i]], label=f'Cluster {i}', 
               alpha=0.6, s=10, edgecolors='none')
ax3.set_title('Agglomerative Clustering', fontsize=14, fontweight='bold')
ax3.set_xlabel('t-SNE Component 1')
ax3.set_ylabel('t-SNE Component 2')
ax3.legend(loc='best', markerscale=2, fontsize=8, ncol=2)
ax3.grid(True, alpha=0.3)

# 4. DBSCAN
ax4 = axes[1, 1]
unique_dbscan_labels = np.unique(dbscan_labels)
colors_dbscan = plt.cm.tab10(np.linspace(0, 1, len(unique_dbscan_labels)))

for i, label in enumerate(unique_dbscan_labels):
    mask = (dbscan_labels == label)
    if label == -1:
        # Noise points in gray
        ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
                   c='gray', label='Noise', 
                   alpha=0.3, s=10, edgecolors='none')
    else:
        ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
                   c=[colors_dbscan[i]], label=f'Cluster {label}', 
                   alpha=0.6, s=10, edgecolors='none')
ax4.set_title('DBSCAN', fontsize=14, fontweight='bold')
ax4.set_xlabel('t-SNE Component 1')
ax4.set_ylabel('t-SNE Component 2')
ax4.legend(loc='best', markerscale=2, fontsize=8, ncol=2)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Detailed Analysis
print("\n" + "="*70)
print("ANALYSIS AND OBSERVATIONS")
print("="*70)

print("\n1. GROUND TRUTH VISUALIZATION:")
print("-" * 70)
print("• The t-SNE plot shows natural groupings of the 10 CIFAR-10 classes")
print("• Some classes are well-separated (e.g., vehicles vs animals)")
print("• Other classes overlap significantly (e.g., cat, dog, deer)")
print("• This indicates the inherent difficulty of the image clustering task")

print("\n2. K-MEANS CLUSTERING:")
print("-" * 70)
print("• K-Means creates relatively compact and spherical clusters")
print("• Cluster boundaries are linear, which may not match the natural")
print("  data structure shown in the ground truth")
print("• Some clusters capture mixed semantic content")
print("• Works best when clusters are roughly spherical and well-separated")

print("\n3. AGGLOMERATIVE CLUSTERING:")
print("-" * 70)
print("• Agglomerative clustering can capture more complex cluster shapes")
print("• Shows hierarchical relationships between data points")
print("• May better follow the natural boundaries visible in ground truth")
print("• Can adapt to the varying density of different regions")

print("\n4. DBSCAN:")
print("-" * 70)
print("• DBSCAN identifies dense regions as clusters")
print("• Marks low-density regions as noise (shown in gray)")
print("• Can discover arbitrary-shaped clusters")
print("• May find fewer clusters than the true number of classes")
print("• Performance depends heavily on the eps and min_samples parameters")

print("\n5. WHICH CLASSES ARE BETTER GROUPED?")
print("-" * 70)
print("By comparing the clustering results with ground truth:")
print("\nWell-separated classes (easier to cluster):")
print("  • Ships - distinct shape and color (water background)")
print("  • Trucks & Automobiles - vehicle features, rectangular shapes")
print("  • Airplanes - distinctive wings and shape")
print("\nOverlapping classes (harder to cluster):")
print("  • Cats, Dogs, Deer - similar animal features, poses")
print("  • Birds & Frogs - small animals with varied appearances")
print("  • Horses - overlap with other animals")

print("\n6. OVERALL COMPARISON:")
print("-" * 70)
print("• K-Means: Fast, creates balanced clusters, but assumes spherical shapes")
print("• Agglomerative: More flexible shapes, captures hierarchy, slower")
print("• DBSCAN: Finds arbitrary shapes, handles noise, but parameter-sensitive")
print("\n• All algorithms struggle with overlapping classes in the feature space")
print("• The t-SNE visualization reveals that some classes are inherently")
print("  difficult to separate using only visual features")

print("\n" + "="*70)
print("HOMEWORK COMPLETED!")
print("="*70)